In [7]:
# =========================
# 📦 1. IMPORTS
# =========================
import pandas as pd
import json

# =========================
# 📁 2. PATHS
# =========================
routes_file = "routes.txt"
trips_file = "trips.txt"
stop_times_file = "stop_times.txt"
stops_file = "stops.txt"

# =========================
# 📥 3. CARGAR DATA
# =========================
routes = pd.read_csv(routes_file)
trips = pd.read_csv(trips_file)
stop_times = pd.read_csv(stop_times_file)
stops = pd.read_csv(stops_file)

# =========================
# 🧹 4. LIMPIEZA
# =========================
routes = routes[['route_id', 'route_short_name', 'route_long_name']]
trips = trips[['route_id', 'trip_id', 'direction_id']]
stop_times = stop_times[['trip_id', 'stop_id', 'stop_sequence']]
stops = stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']]

# =========================
# 🗺️ 5. MAPA DE ESTACIONES
# =========================
stops_map = {
    row['stop_id']: {
        "id": row['stop_id'],
        "nombre": row['stop_name'],
        "lat": row['stop_lat'],
        "lon": row['stop_lon']
    }
    for _, row in stops.iterrows()
}

# =========================
# 🚏 6. AGRUPAR ESTACIONES POR TRIP
# =========================
stop_times_sorted = stop_times.sort_values(by=["trip_id", "stop_sequence"])

trip_stops = {}

for trip_id, group in stop_times_sorted.groupby("trip_id"):
    estaciones = []
    for _, row in group.iterrows():
        stop_id = row["stop_id"]
        if stop_id in stops_map:
            estaciones.append(stops_map[stop_id])
    trip_stops[trip_id] = estaciones

# =========================
# 🧹 7. FUNCION PARA ELIMINAR DUPLICADOS DE ESTACIONES
# =========================
def unique_stops(estaciones):
    seen = set()
    result = []

    for est in estaciones:
        if est["id"] not in seen:
            seen.add(est["id"])
            result.append(est)

    return result

# =========================
# 🚇 8. ARMAR LINEAS (CORRECTO)
# =========================
lineas = {}

# 👇 agrupar por route_id + direction
trips_grouped = trips.groupby(["route_id", "direction_id"])

for (route_id, direction), group in trips_grouped:

    # 👇 tomamos un trip representativo
    trip_id = group.iloc[0]["trip_id"]

    if trip_id not in trip_stops:
        continue

    # info de la ruta
    route_info = routes[routes["route_id"] == route_id].iloc[0]
    linea_nombre = route_info["route_short_name"]

    # 👇 agrupar por nombre real de linea (CLAVE)
    if linea_nombre not in lineas:
        lineas[linea_nombre] = {
            "id": linea_nombre,
            "nombre": f"Línea {linea_nombre}",
            "recorrido": route_info['route_long_name'],
            "ramales": []
        }

    lineas[linea_nombre]["ramales"].append({
        "id": f"{linea_nombre}_{int(direction)}",
        "direccion": int(direction),
        "estaciones": unique_stops(trip_stops[trip_id])
    })

# =========================
# 📦 9. RESULTADO FINAL
# =========================
resultado = {
    "lineas": list(lineas.values())
}



# =========================
# 🔍 10. CREAR INDICE DE BUSQUEDA
# =========================

def normalizar_nombre(nombre):
    return nombre.lower().strip()

indice_estaciones = {}

for linea in resultado["lineas"]:
    for ramal in linea["ramales"]:
        for est in ramal["estaciones"]:
            key = normalizar_nombre(est["nombre"])

            if key not in indice_estaciones:
                indice_estaciones[key] = {
                    "nombre": est["nombre"],
                    "lat": est["lat"],
                    "lon": est["lon"],
                    "lineas": set(),
                    "ids": set()
                }

            indice_estaciones[key]["lineas"].add(linea["id"])
            indice_estaciones[key]["ids"].add(est["id"])

# convertir sets a listas (para JSON)
for k in indice_estaciones:
    indice_estaciones[k]["lineas"] = list(indice_estaciones[k]["lineas"])
    indice_estaciones[k]["ids"] = list(indice_estaciones[k]["ids"])

# agregar al resultado final
resultado["estaciones_index"] = indice_estaciones



# =========================
# 💾 11. GUARDAR JSON
# =========================
output_path = "subtes-static.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)

print(f"✅ JSON generado en {output_path}")

# DEBUG
print(f"Cantidad de líneas: {len(lineas)}")

✅ JSON generado en subtes-static.json
Cantidad de líneas: 8


In [6]:
# =========================
# 🔍 11. CREAR INDICE DE BUSQUEDA
# =========================

def normalizar_nombre(nombre):
    return nombre.lower().strip()

indice_estaciones = {}

for linea in resultado["lineas"]:
    for ramal in linea["ramales"]:
        for est in ramal["estaciones"]:
            key = normalizar_nombre(est["nombre"])

            if key not in indice_estaciones:
                indice_estaciones[key] = {
                    "nombre": est["nombre"],
                    "lat": est["lat"],
                    "lon": est["lon"],
                    "lineas": set(),
                    "ids": set()
                }

            indice_estaciones[key]["lineas"].add(linea["id"])
            indice_estaciones[key]["ids"].add(est["id"])

# convertir sets a listas (para JSON)
for k in indice_estaciones:
    indice_estaciones[k]["lineas"] = list(indice_estaciones[k]["lineas"])
    indice_estaciones[k]["ids"] = list(indice_estaciones[k]["ids"])

# agregar al resultado final
resultado["estaciones_index"] = indice_estaciones